In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import *

%md
### Scenario:
You're joining a large `store_transactions` fact table against a tiny `store_dim` lookup table (in real warehouses, dimension tables like this are often just a handful of rows compared to millions of fact rows). Joining a large table against a small one with a regular shuffle join wastes cluster resources — the small table should be **broadcast** to every executor instead, avoiding a shuffle entirely.

**Problem:**

- Read both files with explicit schemas.
- Join `store_transactions` to `store_dim` on `store_id`, explicitly using the `broadcast()` hint on the small `store_dim` DataFrame (don't just rely on Spark's automatic broadcast threshold — apply the hint yourself).
- Call `.explain()` on the joined DataFrame and confirm you see a `BroadcastHashJoin` (or `BroadcastExchange`) in the physical plan, not a `SortMergeJoin`.
- Aggregate `total_sales` (sum of `amount`) grouped by `store_name` and `region`.
- Order the output by `total_sales` descending.

**store_dim.csv Schema**

| Column | Type |
| :--- | :--- |
| **store_id** | string |
| **store_name** | string |
| **region** | string |

**store_transactions.csv Schema**

| Column | Type |
| :--- | :--- |
| **transaction_id** | string |
| **store_id** | string |
| **amount** | decimal(10,2) |
| **txn_date** | date |

**Expected Output**

| store_name | region | total_sales |
| :--- | :--- | :--- |
| Suburb Store | North | 290.00 |
| Uptown Store | West | 230.00 |
| Downtown Store | East | 195.00 |
| Mall Store | South | 60.00 |

In [0]:
schema_store_dim = StructType(
    [
        StructField("store_id", StringType()),
        StructField("store_name", StringType()),
        StructField("region", StringType()),
    ]
)

schema_store_txns = StructType(
    [
        StructField("transaction_id", StringType()),
        StructField("store_id", StringType()),
        StructField("amount", DecimalType()),
        StructField("txn_date", DateType()),
    ]
)

store_dim_df = (
    spark.read.format("csv").option("header", True).schema(schema_store_dim).load("/Workspace/Users/jeevan.busi8008@gmail.com/spark-practice/data/store_dim.csv")
)

store_txns_df = (
    spark.read.format("csv").option("header", True).schema(schema_store_txns).load("/Workspace/Users/jeevan.busi8008@gmail.com/spark-practice/data/store_transactions.csv")
)

store_joined_df = (
    store_txns_df.join(broadcast(store_dim_df), "store_id", "inner")
)
store_joined_df.explain() ## To verify broadcast join applied or not 

total_sales_df = (
    store_joined_df.groupBy(col("store_name"), col("region"))
    .agg(sum("amount").alias("total_sales"))
)
total_sales_df.orderBy(col("total_sales").desc()).show()

### Scenario:
Your `customer_dim` table needs to preserve **history** of attribute changes (e.g., a customer moving regions), not just overwrite them — this is a classic **SCD Type 2** dimension. Each row has an `effective_start_date`, `effective_end_date`, and `is_current` flag. When a tracked attribute changes, the old row must be "closed out" (end-dated, `is_current = false`) and a new row inserted as the current version. If nothing about a customer changed, no new row should be created.

**Problem:**

- Load `customer_scd2_seed.csv` and write it as a managed Delta table named `customer_scd2` (one-time setup representing the current state of history so far).
- Read `customer_scd2_updates.csv` — this represents today's snapshot (`as_of_date = 2024-03-01`) of customer attributes from the source system, with no history columns.
- Compare each incoming row's `region` against that customer's **current** (`is_current = true`) row in `customer_scd2`:
  - If `region` is unchanged, do nothing for that customer.
  - If `region` has changed, **expire** the existing current row (set `effective_end_date = 2024-03-01`, `is_current = false`) and **insert** a new row with the updated `region`, `effective_start_date = 2024-03-01`, `effective_end_date = null`, `is_current = true`.
  - If the `customer_id` doesn't exist in `customer_scd2` at all, insert it as a brand-new current row with `effective_start_date = 2024-03-01`.
- Display the final `customer_scd2` table, sorted by `customer_id` ascending, then `effective_start_date` ascending.

**customer_scd2_seed.csv Schema**

| Column | Type |
| :--- | :--- |
| **customer_id** | string |
| **customer_name** | string |
| **region** | string |
| **effective_start_date** | date |
| **effective_end_date** | date |
| **is_current** | boolean |

**customer_scd2_updates.csv Schema**

| Column | Type |
| :--- | :--- |
| **customer_id** | string |
| **customer_name** | string |
| **region** | string |

**Expected Output — final `customer_scd2` table**

| customer_id | customer_name | region | effective_start_date | effective_end_date | is_current |
| :--- | :--- | :--- | :--- | :--- | :--- |
| C001 | Alice Johnson | West | 2024-01-01 | null | true |
| C002 | Bob Singh | East | 2024-01-01 | 2024-03-01 | false |
| C002 | Bob Singh | North | 2024-03-01 | null | true |
| C003 | Carol Mehta | East | 2024-01-01 | null | true |
| C004 | David Kim | South | 2024-03-01 | null | true |

In [0]:
%sql
select * from pyspark_practice.default.customer_dim

In [0]:
## Loading sustomer seed data
schema_cust_scd2 = StructType(
    [
        StructField("customer_id", StringType()),
        StructField("customer_name", StringType()),
        StructField("region", StringType()),
        StructField("effective_start_date", DateType()),
        StructField("effective_end_date", DateType()),
        StructField("is_current", BooleanType())
    ]
)

cust_seed_df = (
    spark.read.format("csv").option("header", True).schema(schema_cust_scd2).load("/Workspace/Users/jeevan.busi8008@gmail.com/spark-practice/data/customer_scd2_seed.csv")
)
## since it is onetime seed data mode is overwite to avoid duplicates 
cust_seed_df.write.format("delta").mode("overwrite").saveAsTable("pyspark_practice.default.customer_dim")

## Transforming incoming data with SCD type 2 logic
schema_cust_incoming = StructType(
    [
        StructField("customer_id", StringType()),
        StructField("customer_name", StringType()),
        StructField("region", StringType())
    ]
)
cust_incoming_df = (
    spark.read.format("csv").option("header", True).schema(schema_cust_incoming).load("/Workspace/Users/jeevan.busi8008@gmail.com/spark-practice/data/customer_scd2_updates.csv")
)

cust_scd2_target_df = DeltaTable.forName(spark, "pyspark_practice.default.customer_dim")
active_cust_scd2_target_df = cust_scd2_target_df.toDF().filter(col("is_current")==True)

## Changed cust data
changed_cust_df = (
    cust_incoming_df.alias("s").join(
        active_cust_scd2_target_df.alias("t"), 
        "customer_id", "inner"
    ).filter(
        (col("t.is_current") == True)
        & (col("t.region") != col("s.region"))
    )
)

(
    cust_scd2_target_df.alias("t").merge(
        changed_cust_df.alias("s"), "t.customer_id=s.customer_id"
    )
    .whenMatchedUpdate(
        set={
            "effective_end_date": lit("2024-03-01").cast("date"),
            "is_current": lit(False)
        }
    ).execute()
)

audit_cols_existing_cust_df = (
    changed_cust_df.select(
        col("customer_id"),
        col("s.region").alias("region"),
        lit("2024-03-01").cast(DateType()).alias("effective_start_date"),
        lit(None).cast(DateType()).alias("effective_end_date"),
        lit(True).alias("is_current")
    )
)
audit_cols_existing_cust_df.write.format("delta").mode("append").saveAsTable("pyspark_practice.default.customer_dim")

## New cust data
new_cust_df = cust_incoming_df.join(active_cust_scd2_target_df, "customer_id", "left_anti")
audit_cols_new_cust_df = (
    new_cust_df.withColumns(
        {
            "effective_start_date": lit("2024-03-01").cast("date"),
            "is_current": lit(True)
        }
    )
)
audit_cols_new_cust_df.write.format("delta").mode("append").saveAsTable("pyspark_practice.default.customer_dim")

